# Session 1 — Your turn: Attention inside a real model

Write one attention head the way a decoder computes it, check it against PyTorch's own implementation, and see why the scores are divided by the square root of the key size.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
import torch
import torch.nn.functional as F

rng = np.random.default_rng(0)
T, d_model, d_head = 6, 8, 4
x = rng.normal(size=(T, d_model))
Wq, Wk, Wv = (rng.normal(size=(d_model, d_head)) for _ in range(3))
from aij_check import check, close, summary

## Task 1: One causal attention head

Write `attention(x, Wq, Wk, Wv)` returning `(weights, output)`: queries, keys and values are `x @ Wq`, `x @ Wk`, `x @ Wv`; the scores are `Q @ K.T / sqrt(d_head)`;
positions after the current one are blocked (set to minus infinity) before a softmax over each row; the output is `weights @ V`.

In [2]:
def attention(x, Wq, Wk, Wv):
    Q, K, V = x @ Wq, x @ Wk, x @ Wv
    scores = Q @ K.T / np.sqrt(K.shape[1])
    scores = np.where(np.tril(np.ones_like(scores, dtype=bool)), scores, -np.inf)
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = e / e.sum(axis=1, keepdims=True)
    return weights, weights @ V


weights, output = attention(x, Wq, Wk, Wv)

In [3]:
reference = F.scaled_dot_product_attention(*(torch.tensor(a)[None] for a in (x @ Wq, x @ Wk, x @ Wv)), is_causal=True)[0].numpy()
check("each row of weights sums to 1", lambda: close(weights.sum(axis=1), 1.0, 1e-9))
check("no weight points at a later token", lambda: np.allclose(np.triu(weights, 1), 0))
check("the first token reads only itself", lambda: close(weights[0], [1, 0, 0, 0, 0, 0], 1e-9))
check("the output agrees with PyTorch's scaled_dot_product_attention", lambda: close(output, reference, 1e-9))
summary()

PASS  each row of weights sums to 1
PASS  no weight points at a later token
PASS  the first token reads only itself
PASS  the output agrees with PyTorch's scaled_dot_product_attention

4 of 4 checks passed.  Well done.


**Why it works.** This is the whole computation inside one head of GPT-2; the session checks the same thing against the real model to within float rounding (1e-6). The masked entries become exactly 0 after the softmax.

## Task 2: Why divide by the square root of d_k?

For key sizes `d_k` of 4, 64 and 1024, draw 500 random queries and 32 random keys each (standard normal), compute the softmax of the scores with and without the division by `sqrt(d_k)`, and store in `largest_weight` the **average largest weight** per case:
`largest_weight[(d_k, "unscaled")]` and `largest_weight[(d_k, "scaled")]`. Use `np.random.default_rng(1)`.

In [4]:
def softmax(s):
    e = np.exp(s - s.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


gen = np.random.default_rng(1)
largest_weight = {}
for d_k in (4, 64, 1024):
    q, k = gen.normal(size=(500, 1, d_k)), gen.normal(size=(500, 32, d_k))
    raw = (q * k).sum(-1)
    largest_weight[(d_k, "unscaled")] = softmax(raw).max(-1).mean()
    largest_weight[(d_k, "scaled")] = softmax(raw / np.sqrt(d_k)).max(-1).mean()
print({k: round(v, 3) for k, v in largest_weight.items()})

{(4, 'unscaled'): np.float64(0.325), (4, 'scaled'): np.float64(0.152), (64, 'unscaled'): np.float64(0.807), (64, 'scaled'): np.float64(0.159), (1024, 'unscaled'): np.float64(0.956), (1024, 'scaled'): np.float64(0.167)}


In [5]:
check("six numbers", lambda: len(largest_weight) == 6)
check("without the division the largest weight climbs towards 1 as d_k grows", lambda: largest_weight[(1024, "unscaled")] > 0.9 > largest_weight[(4, "unscaled")] + 0.3)
check("with the division it stays roughly constant", lambda: max(largest_weight[(d, "scaled")] for d in (4, 64, 1024)) - min(largest_weight[(d, "scaled")] for d in (4, 64, 1024)) < 0.08)
check("at d_k = 4 the division matters least", lambda: largest_weight[(4, "unscaled")] - largest_weight[(4, "scaled")] < 0.5 * (largest_weight[(1024, "unscaled")] - largest_weight[(1024, "scaled")]))
summary()

PASS  six numbers
PASS  without the division the largest weight climbs towards 1 as d_k grows
PASS  with the division it stays roughly constant
PASS  at d_k = 4 the division matters least

4 of 4 checks passed.  Well done.


**Why it works.** A dot product of two random vectors has a spread that grows with sqrt(d_k). Without the division the softmax sees huge differences and behaves like a hard choice (and its gradient vanishes); dividing keeps the spread near 1 whatever the head size.